# Corroborated same-band mask propagation for all reviewed dippers

Sidecar diagnostic only: compare the current production masked per-camera GP with the same pipeline after propagating connected event intervals corroborated by at least two cameras. Production source is not modified.

In [1]:
from contextlib import contextmanager
import json
from pathlib import Path
import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next((p for p in candidate_roots if (p / 'pyproject.toml').is_file() and (p / 'malca' / 'core' / 'baseline.py').is_file()), None)
if repo_root is None:
    raise RuntimeError(f'Could not find the MALCA repository above {Path.cwd()}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import malca.core.baseline as baseline_module
from malca.config import (
    GP_BRIGHT_SIGMA_THRESH, GP_DIP_SIGMA_THRESH, GP_MIN_GP_POINTS, GP_PAD_DAYS,
    GP_STIFF_MIN_DAYS, GP_STIFF_SCALE_FRACTION,
)
from malca.core.utils import clean_lc
from malca.io.lightcurve_io import load_lightcurve_df, to_asassn_algorithm_frame
from malca.plotting.lightcurve_publication import FIG_SINGLE_COL_WIDTH, PUBLICATION_STYLE, finalize_publication_figure
from malca.review.native_lightcurve import resolve_lightcurve_path
from malca.review.store import get_candidate_payload
from malca.stv.events import DEFAULT_BASELINE_KWARGS, score_events_bayesian

JD_OFFSET = 2458000.0
RUN_ROOT = repo_root / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4'
RUN_PARAMS = json.loads((RUN_ROOT / 'run_params.json').read_text())
REVIEW_DB = RUN_ROOT / 'review' / 'review.db'
OUTPUT_DIR = repo_root / 'output' / 'pdf' / 'baseline_methods' / 'diagnostics' / 'reviewed_dippers'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = OUTPUT_DIR / 'corroborated_mask_propagation_reviewed_dippers.pdf'
MANIFEST_PATH = OUTPUT_DIR / 'corroborated_mask_propagation_reviewed_dippers.csv'
CAMERA_PALETTE = ('#0072B2', '#009E73', '#56B4E9', '#6A3D9A', '#7F7F7F', '#E69F00', '#332288', '#44AA99', '#999933', '#8C6D31', '#333333', '#117733')

COHORT_SQL = '''
SELECT c.candidate_id, c.asas_sn_id
FROM candidates AS c
INNER JOIN reviews AS r ON r.candidate_id = c.candidate_id
WHERE lower(trim(coalesce(r.event_class, ''))) = ?
ORDER BY c.candidate_id
'''

/opt/homebrew/Caskroom/miniconda/base/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_reviewed_dippers():
    read_only_uri = f'file:{REVIEW_DB.as_posix()}?mode=ro'
    with sqlite3.connect(read_only_uri, uri=True) as connection:
        cohort = pd.read_sql_query(COHORT_SQL, connection, params=('dipper',))
        paths = []
        for candidate_id in cohort['candidate_id'].astype(str):
            payload = get_candidate_payload(connection, candidate_id)
            resolved = resolve_lightcurve_path(payload, RUN_ROOT)
            paths.append(str(resolved.resolve()) if resolved is not None else None)
    cohort['resolved_lightcurve_path'] = paths
    if cohort['candidate_id'].duplicated().any():
        raise AssertionError('Reviewed-dipper query returned duplicate candidate IDs.')
    return cohort

def select_best_band(cleaned):
    bands = pd.to_numeric(cleaned['v_g_band'], errors='coerce')
    counts = bands.value_counts()
    candidates = [band for band in (0.0, 1.0) if counts.get(band, 0) > 0]
    if not candidates:
        raise ValueError('No recognized g or V band observations.')
    band_value = max(candidates, key=lambda band: (int(counts.get(band, 0)), band == 0.0))
    return band_value, {0.0: 'g', 1.0: 'V'}[band_value]

def prepare_lightcurve(candidate_row):
    path_value = candidate_row['resolved_lightcurve_path']
    if not path_value or not Path(path_value).is_file():
        raise FileNotFoundError(path_value)
    canonical = load_lightcurve_df(Path(path_value), apply_quality=True)
    cleaned = clean_lc(to_asassn_algorithm_frame(canonical)).reset_index(drop=True)
    band_value, band_label = select_best_band(cleaned)
    bands = pd.to_numeric(cleaned['v_g_band'], errors='coerce')
    lightcurve = cleaned.loc[np.isclose(bands, band_value)].copy().reset_index(drop=True)
    return lightcurve, band_label

cohort = load_reviewed_dippers()
if len(cohort) != 183:
    raise AssertionError(f'Expected 183 reviewed dippers, found {len(cohort)}.')
print(f'Reviewed dippers: {len(cohort)}')

Reviewed dippers: 183


In [3]:
def initial_intervals_by_camera(lightcurve):
    intervals_by_camera = {}
    for camera, sub in lightcurve.groupby('camera#', sort=True):
        ordered = sub.sort_values('JD')
        t = ordered['JD'].to_numpy(float)
        mag = ordered['mag'].to_numpy(float)
        error = ordered['error'].to_numpy(float)
        finite = np.isfinite(t) & np.isfinite(mag)
        median_mag = float(np.nanmedian(mag[finite]))
        base_rough = baseline_module._compute_base_rough(
            t, mag, error, finite, median_mag, auto_scale_gp=True,
            min_gp_points=GP_MIN_GP_POINTS, stiff_scale_fraction=GP_STIFF_SCALE_FRACTION,
            stiff_min_days=GP_STIFF_MIN_DAYS, S0=DEFAULT_BASELINE_KWARGS['S0'], q=DEFAULT_BASELINE_KWARGS['q'],
        )
        flags, _, _, _ = baseline_module._mask_excursions(
            t, mag, base_rough, finite, median_mag, mag_err=error,
            dip_sigma_thresh=GP_DIP_SIGMA_THRESH, bright_sigma_thresh=GP_BRIGHT_SIGMA_THRESH, pad_days=GP_PAD_DAYS,
        )
        flagged_indices = np.flatnonzero(flags)
        runs = np.split(flagged_indices, np.flatnonzero(np.diff(flagged_indices) > 1) + 1) if flagged_indices.size else []
        intervals_by_camera[camera] = [(float(t[run[0]]) - GP_PAD_DAYS, float(t[run[-1]]) + GP_PAD_DAYS) for run in runs if run.size]
    return intervals_by_camera

def corroborated_intervals(intervals_by_camera, min_cameras=2):
    tagged = sorted((float(lo), float(hi), camera) for camera, intervals in intervals_by_camera.items() for lo, hi in intervals)
    if not tagged:
        return []
    components = []
    current_lo, current_hi, current_camera = tagged[0]
    current_cameras = {current_camera}
    for lo, hi, camera in tagged[1:]:
        if lo <= current_hi:
            current_hi = max(current_hi, hi)
            current_cameras.add(camera)
        else:
            if len(current_cameras) >= int(min_cameras):
                components.append((current_lo, current_hi))
            current_lo, current_hi, current_cameras = lo, hi, {camera}
    if len(current_cameras) >= int(min_cameras):
        components.append((current_lo, current_hi))
    return components

original_mask_excursions = baseline_module._mask_excursions

@contextmanager
def propagate_intervals(extra_intervals):
    def mask_with_propagation(*args, **kwargs):
        flags, keep, intervals, s0 = original_mask_excursions(*args, **kwargs)
        t = np.asarray(args[0], float)
        added = np.zeros(len(t), dtype=bool)
        for lo, hi in extra_intervals:
            added |= (t >= float(lo)) & (t <= float(hi))
        flags = np.asarray(flags, bool) | added
        keep = np.asarray(keep, bool) & ~added
        intervals = baseline_module._union_intervals(list(intervals) + list(extra_intervals))
        return flags, keep, intervals, s0
    baseline_module._mask_excursions = mask_with_propagation
    try:
        yield
    finally:
        baseline_module._mask_excursions = original_mask_excursions

def run_comparison(lightcurve):
    current = baseline_module.per_camera_gp_baseline_masked(lightcurve, **DEFAULT_BASELINE_KWARGS).reset_index(drop=True)
    camera_intervals = initial_intervals_by_camera(lightcurve)
    shared_intervals = corroborated_intervals(camera_intervals, min_cameras=2)
    with propagate_intervals(shared_intervals):
        propagated = baseline_module.per_camera_gp_baseline_masked(lightcurve, **DEFAULT_BASELINE_KWARGS).reset_index(drop=True)
    return current, propagated, shared_intervals

EVENT_SCORING_KWARGS = {
    'p_points': int(RUN_PARAMS['p_points']),
    'mag_points': int(RUN_PARAMS['mag_points']),
    'trigger_mode': str(RUN_PARAMS['trigger_mode']),
    'significance_threshold': float(RUN_PARAMS['significance_threshold']),
    'run_min_points': int(RUN_PARAMS['run_min_points']),
    'max_gap_points': int(RUN_PARAMS['run_max_gap_points']),
    'run_max_gap_days': RUN_PARAMS['run_max_gap_days'],
    'run_min_duration_days': RUN_PARAMS['run_min_duration_days'],
    'compute_event_prob': True,
}

def production_event_run_mask(lightcurve, baseline_result):
    event_mask = np.zeros(len(lightcurve), dtype=bool)
    counts = {}
    for kind in ('dip', 'jump'):
        score = score_events_bayesian(
            lightcurve, kind=kind, baseline_func=None, df_base=baseline_result,
            logbf_threshold=float(RUN_PARAMS[f'logbf_threshold_{kind}']),
            **EVENT_SCORING_KWARGS,
        )
        indices = np.asarray(score['event_indices'], dtype=int)
        if indices.size:
            event_mask[indices] = True
        counts[kind] = int(indices.size)
    counts['either'] = int(event_mask.sum())
    return event_mask, counts

In [4]:
def camera_colors(lightcurve):
    cameras = sorted(lightcurve['camera#'].dropna().unique(), key=str)
    return {camera: CAMERA_PALETTE[i % len(CAMERA_PALETTE)] for i, camera in enumerate(cameras)}

def plot_baseline_panel(ax, frame, colors, title):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        x = sub['JD'].to_numpy(float) - JD_OFFSET
        color = colors[camera]
        ax.scatter(x, sub['mag'], s=5, color=color, edgecolors='black', linewidths=0.18, zorder=2)
        ax.plot(x, sub['baseline'], color=color, linewidth=1.1, zorder=3)
        event_run = sub['event_run'].fillna(False).to_numpy(bool)
        if event_run.any():
            ax.scatter(x[event_run], sub.loc[event_run, 'mag'], s=13, marker='x', color='#b2182b', linewidths=0.7, zorder=5)
    ax.invert_yaxis()
    ax.set_ylabel(r'$m_i$ [mag]')
    ax.set_title(title, fontsize=8, pad=3)

def plot_residual_panel(ax, frame, colors, title):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        x = sub['JD'].to_numpy(float) - JD_OFFSET
        color = colors[camera]
        ax.scatter(x, sub['resid'], s=5, color=color, edgecolors='black', linewidths=0.18, zorder=2)
        event_run = sub['event_run'].fillna(False).to_numpy(bool)
        if event_run.any():
            ax.scatter(x[event_run], sub.loc[event_run, 'resid'], s=13, marker='x', color='#b2182b', linewidths=0.7, zorder=5)
    ax.axhline(0, color='0.25', linewidth=0.7)
    ax.invert_yaxis()
    ax.set_ylabel(r'$r_i$ [mag]')
    ax.set_title(title, fontsize=8, pad=3)

def make_page(candidate_row, band_label, lightcurve, current, propagated, current_event_mask, propagated_event_mask):
    colors = camera_colors(lightcurve)
    current = current.copy()
    propagated = propagated.copy()
    current['event_run'] = np.asarray(current_event_mask, dtype=bool)
    propagated['event_run'] = np.asarray(propagated_event_mask, dtype=bool)
    with plt.rc_context(PUBLICATION_STYLE):
        fig, axes = plt.subplots(4, 1, figsize=(FIG_SINGLE_COL_WIDTH, 8.0), sharex=True)
        plot_baseline_panel(axes[0], current, colors, 'Current masked per-camera GP')
        plot_baseline_panel(axes[1], propagated, colors, 'Corroborated-mask per-camera GP')
        plot_residual_panel(axes[2], current, colors, 'Current residual')
        plot_residual_panel(axes[3], propagated, colors, 'Propagated-mask residual')
        axes[1].set_ylim(axes[0].get_ylim())
        residual_values = np.concatenate([current['resid'].to_numpy(float), propagated['resid'].to_numpy(float)])
        residual_values = residual_values[np.isfinite(residual_values)]
        residual_limit = max(0.15, float(np.max(np.abs(residual_values))) * 1.05)
        axes[2].set_ylim(residual_limit, -residual_limit)
        axes[3].set_ylim(residual_limit, -residual_limit)
        axes[-1].set_xlabel('JD - 2458000')
        for ax in axes:
            ax.tick_params(direction='in', top=True, right=True)
        source_id = candidate_row.get('asas_sn_id')
        if pd.isna(source_id) or not str(source_id).strip():
            source_id = candidate_row['candidate_id']
        fig.suptitle(f'ASAS-SN {str(source_id).strip()}, {band_label} band', fontsize=8)
        finalize_publication_figure(fig, rect=(0, 0, 1, 0.982))
        return fig

In [5]:
manifest_rows = []
with PdfPages(PDF_PATH) as atlas_pdf:
    for position, (_, candidate_row) in enumerate(cohort.iterrows(), start=1):
        candidate_id = str(candidate_row['candidate_id'])
        print(f'[{position:03d}/{len(cohort):03d}] {candidate_id}', end=' ... ')
        try:
            lightcurve, band_label = prepare_lightcurve(candidate_row)
            current, propagated, shared_intervals = run_comparison(lightcurve)
            current_event_mask, current_event_counts = production_event_run_mask(lightcurve, current)
            propagated_event_mask, propagated_event_counts = production_event_run_mask(lightcurve, propagated)
            figure = make_page(
                candidate_row, band_label, lightcurve, current, propagated,
                current_event_mask, propagated_event_mask,
            )
            atlas_pdf.savefig(figure)
            plt.close(figure)
            initial_masked = int(current['is_masked'].fillna(False).sum())
            propagated_masked = int(propagated['is_masked'].fillna(False).sum())
            manifest_rows.append({
                'event_class': 'dipper', 'candidate_id': candidate_id, 'asas_sn_id': candidate_row.get('asas_sn_id'),
                'lightcurve_path': candidate_row.get('resolved_lightcurve_path'), 'band': band_label,
                'n_points': int(len(lightcurve)), 'n_cameras': int(lightcurve['camera#'].nunique()),
                'initial_masked': initial_masked, 'propagated_masked': propagated_masked,
                'added_masked': propagated_masked - initial_masked,
                'initial_event_run_points': current_event_counts['either'],
                'initial_dip_run_points': current_event_counts['dip'],
                'initial_jump_run_points': current_event_counts['jump'],
                'propagated_event_run_points': propagated_event_counts['either'],
                'propagated_dip_run_points': propagated_event_counts['dip'],
                'propagated_jump_run_points': propagated_event_counts['jump'],
                'initial_mask_fraction': initial_masked / len(lightcurve),
                'propagated_mask_fraction': propagated_masked / len(lightcurve),
                'corroborated_intervals': int(len(shared_intervals)),
                'corroborated_duration_days': float(sum(hi - lo for lo, hi in shared_intervals)),
                'initial_consensus_cameras': int(current.groupby('camera#')['needs_consensus'].any().sum()),
                'propagated_consensus_cameras': int(propagated.groupby('camera#')['needs_consensus'].any().sum()),
                'initial_sources': ','.join(sorted(current['baseline_source'].dropna().astype(str).unique())),
                'propagated_sources': ','.join(sorted(propagated['baseline_source'].dropna().astype(str).unique())),
                'status': 'ok', 'error': '',
            })
            print(
                f'{len(lightcurve)} points, {lightcurve["camera#"].nunique()} cameras, '
                f'{current_event_counts["either"]} -> {propagated_event_counts["either"]} event-run points'
            )
        except Exception as exc:
            manifest_rows.append({
                'event_class': 'dipper', 'candidate_id': candidate_id, 'asas_sn_id': candidate_row.get('asas_sn_id'),
                'lightcurve_path': candidate_row.get('resolved_lightcurve_path'), 'band': '',
                'n_points': 0, 'n_cameras': 0, 'initial_masked': 0, 'propagated_masked': 0, 'added_masked': 0,
                'initial_event_run_points': 0, 'initial_dip_run_points': 0, 'initial_jump_run_points': 0,
                'propagated_event_run_points': 0, 'propagated_dip_run_points': 0, 'propagated_jump_run_points': 0,
                'initial_mask_fraction': np.nan, 'propagated_mask_fraction': np.nan,
                'corroborated_intervals': 0, 'corroborated_duration_days': 0.0,
                'initial_consensus_cameras': 0, 'propagated_consensus_cameras': 0,
                'initial_sources': '', 'propagated_sources': '',
                'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
            })
            print(f'FAILED: {type(exc).__name__}: {exc}')

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False)
display(manifest['status'].value_counts().rename_axis('status').to_frame('candidates'))
display(manifest.groupby('band').size().rename('candidates').to_frame())
display(manifest['added_masked'].describe().to_frame())
print(PDF_PATH)
print(MANIFEST_PATH)
failures = manifest.loc[manifest['status'] != 'ok']
if not failures.empty:
    display(failures)
    raise RuntimeError(f'{len(failures)} reviewed-dipper candidates failed.')
if len(manifest) != 183:
    raise AssertionError(f'Expected 183 manifest rows, got {len(manifest)}.')

[001/183] stv_103079502524 ... 

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


740 points, 2 cameras, 2 -> 2 event-run points
[002/183] stv_103080542701 ... 

687 points, 2 cameras, 0 -> 0 event-run points
[003/183] stv_111669273145 ... 

1514 points, 4 cameras, 6 -> 6 event-run points
[004/183] stv_111669291649 ... 

...

563 points, 5 cameras, 4 -> 4 event-run points


,candidates
status,
ok,183


,candidates
band,
g,183


,added_masked
count,183.000000
mean,50.912568
std,84.965300
min,-86.000000
25%,7.000000
50%,24.000000
75%,56.000000
max,684.000000


/Users/calder/code/malca/output/pdf/baseline_methods/diagnostics/reviewed_dippers/corroborated_mask_propagation_reviewed_dippers.pdf
/Users/calder/code/malca/output/pdf/baseline_methods/diagnostics/reviewed_dippers/corroborated_mask_propagation_reviewed_dippers.csv
